# Infinite-Width Uncertainty: results walkthrough

This notebook reproduces the four-way comparison from the README on the
scikit-learn diabetes dataset. It fits an **NNGP** (the infinite-width
limit of a fully-connected network, as an exact Gaussian process), a
**deep ensemble** (the finite-width counterpart), **MC-dropout**, and a
textbook **RBF-GP**, then measures the quality of their *uncertainty* on
an i.i.d. split and on a covariate-shift split.

Run top to bottom. On a laptop CPU the whole thing takes a couple of
minutes; the neural baselines dominate the runtime.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, pandas as pd
from iwuq import data, metrics, plotting
from iwuq.models import MODEL_REGISTRY
from iwuq.selective import aurc
SEED = 0

## 1. Data and the two splits

The random split is i.i.d. The covariate-shift split trains on the lower
80% of BMI and tests on the upper 20% tail, so the test inputs sit beyond
the training support in that direction.

In [ ]:
ds = data.load_diabetes_dataset()
print('X', ds.X.shape, '| y', ds.y.shape)
print('features:', ds.feature_names)
splits = {
    'random': data.random_split(ds, test_frac=0.2, seed=SEED),
    'shift':  data.covariate_shift_split(ds, 'bmi', test_frac=0.2),
}
for k, s in splits.items():
    print(k, '-> train', len(s.y_train), 'test', len(s.y_test), '|', s.kind)

## 2. Fit all four models on both splits

Every model exposes the same `fit` / `predict_dist` interface, so the loop
is identical across methods. `predict_dist` returns a per-point mean and a
standard deviation of the predictive distribution over a new observation.

In [ ]:
def fit_all(split):
    preds = {}
    for name, cls in MODEL_REGISTRY.items():
        model = cls(seed=SEED) if 'seed' in cls.__init__.__code__.co_varnames else cls()
        model.fit(split.X_train, split.y_train)
        preds[name] = model.predict_dist(split.X_test)
    return preds

preds_by_split = {k: (fit_all(s), s.y_test) for k, s in splits.items()}
print('done')

## 3. Metrics

Point accuracy (RMSE, MAE), distributional quality (NLL, CRPS), interval
calibration (95% coverage, mean width, miscalibration area) and selective
prediction (AURC). Lower is better for everything except coverage, which
should track the nominal 0.95.

In [ ]:
def metrics_table(preds, y):
    rows = {}
    for name, (mean, std) in preds.items():
        row = metrics.evaluate(y, mean, std)
        row['AURC'] = aurc(y, mean, std)
        rows[name] = row
    return pd.DataFrame(rows).T.round(3)

for k, (preds, y) in preds_by_split.items():
    print(f'\n=== {k} split ===')
    print(metrics_table(preds, y).to_string())

## 4. Figures

Reliability diagrams, the uncertainty in/out-of-support comparison, and a
risk-coverage curve.

In [ ]:
p_rand, y_rand = preds_by_split['random']
p_shift, y_shift = preds_by_split['shift']
plotting.plot_calibration(p_rand, y_rand, title='Calibration -- random');
plotting.plot_calibration(p_shift, y_shift, title='Calibration -- shift');
plotting.plot_uncertainty_under_shift(p_rand, y_rand, p_shift, y_shift);
plotting.plot_risk_coverage(p_shift, y_shift, title='Risk-coverage -- shift');

## 5. Reading the results

See the README for the full discussion. In brief, for this dataset and
seed: in-support the calibrated NNGP matches the tuned RBF-GP and ranks
errors best (lowest AURC); under covariate shift every model degrades, but
the GP-based methods and the deep ensemble stay far better calibrated than
MC-dropout, which becomes badly overconfident. Numbers are from a single
seed -- average over several before drawing firm conclusions.